# Select n user ids; create a train ds where parquets don't have any users outside selected ids 

In [21]:
from typing import List
import os

import pandas as pd
from tqdm import tqdm


N_USERS = 15000


def get_k_train_user_ids(k: int, ds_path: str = 'data/train_trx_file.parquet') -> List[int]:
    user_ids = []
    for file_name in tqdm(os.listdir(ds_path)):
        if not file_name.endswith('.parquet'):
            continue
        file_path = os.path.join(ds_path, file_name)
        df = pd.read_parquet(file_path)
        user_ids.extend(df['encoded_client_id'].unique())
        if len(user_ids) > k:
            break
    return user_ids[:k]


def create_subdataset(out_path: str, 
                      user_ids: List[int], 
                      in_path: str) -> None:
    os.makedirs(out_path)
    for file_name in tqdm(os.listdir(in_path)):
        if not file_name.endswith('.parquet'):
            continue
        file_path = os.path.join(in_path, file_name)
        df = pd.read_parquet(file_path)
        df = df[df['user_id'].isin(user_ids)]
        if not df.empty:
            df.to_parquet(os.path.join(out_path, file_name))


In [22]:
user_ids = get_k_train_user_ids(N_USERS)

  5%|▌         | 87/1602 [00:00<00:07, 191.27it/s]


In [86]:
create_subdataset(out_path=f'data/train_trx_file_{len(user_ids)}.parquet', 
                  user_ids=user_ids, 
                  in_path='data/train_trx_file.parquet')

100%|██████████| 1602/1602 [00:15<00:00, 104.06it/s]


# Create graph without selected ids

In [68]:
import dgl
import torch



GRAPH_FILE_PATH = 'data/graphs/weighted_train_graph_has_test_items/train_graph.bin'
ITEM_ID2TRAIN_GRAPH_ID_ORIG = torch.load('data/graphs/weighted_train_graph_has_test_items/item_id2train_graph_id.pt') 
CLIENT_ID2TRAIN_GRAPH_ID_ORIG = torch.load('data/graphs/weighted_train_graph_has_test_items/client_id2train_graph_id.pt')

(graph,), _ = dgl.load_graphs(GRAPH_FILE_PATH, [0])

# Получить мапы client_id2train_graph_id, item_id2train_graph_id 
client_train_graph_ids = CLIENT_ID2TRAIN_GRAPH_ID_ORIG[torch.tensor(user_ids)]

nodes_of_interest = torch.concat([client_train_graph_ids, ITEM_ID2TRAIN_GRAPH_ID_ORIG[2:]])

subgraph = dgl.node_subgraph(graph, nodes=nodes_of_interest)

In [69]:
def invert_tensor_based_map(torch_map, invalid_index_val = -1) -> torch.Tensor:
    inverted_map = torch.full((max(torch_map) + 1,), invalid_index_val)
    inverted_map[torch_map] = torch.arange(torch_map.shape[0])
    return inverted_map

In [70]:
# CLIENT_ID2TRAIN_GRAPH_ID_ORIG_DICT = { k: int(v) for k, v in enumerate(CLIENT_ID2TRAIN_GRAPH_ID_ORIG) if k==0 or v!=0}

In [71]:
graph_id_to_subgraph_id = invert_tensor_based_map(subgraph.ndata['_ID'])

In [72]:
graph_id_to_subgraph_id

tensor([    -1,     -1,     -1,  ..., 211676, 211679, 211690])

In [73]:
client_id2train_subgraph_id = graph_id_to_subgraph_id[CLIENT_ID2TRAIN_GRAPH_ID_ORIG]

In [74]:
print(client_id2train_subgraph_id.unique().shape)

torch.Size([15001])


In [75]:
torch.equal(client_id2train_subgraph_id[user_ids], torch.arange(N_USERS))

True

In [76]:
item_id2train_subgraph_id = graph_id_to_subgraph_id[ITEM_ID2TRAIN_GRAPH_ID_ORIG]

In [78]:
n_urls = 196693

n_urls + N_USERS

211693

In [81]:
torch.equal(item_id2train_subgraph_id[2:], torch.arange(N_USERS, N_USERS+n_urls))

True

In [83]:
###### Saving ######

subgraph_graph_dir_path = f'data/graphs/weighted_subgraph_for_test_with_{N_USERS}_users_and_all_items/'

os.makedirs(subgraph_graph_dir_path)

torch.save(item_id2train_subgraph_id, os.path.join(subgraph_graph_dir_path, 'item_id2train_graph_id.pt'))
torch.save(client_id2train_subgraph_id, os.path.join(subgraph_graph_dir_path, 'client_id2train_graph_id.pt'))
dgl.save_graphs(os.path.join(subgraph_graph_dir_path, 'train_graph.bin'), [subgraph])

# Create a graph_item_id_to_llm_embedding for this sub-dataset

we don't need to create that currently, beacause we are using all items